# 15 · Layered 3D access maps with deck.gl

**Question:** Can an interactive map compare access scenarios without confusing people with metres?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
The initial baseline uses **synthetic educational fixtures** near a Gaborone-area anchor.
Extended investigations explicitly identify bundled real OSM, Sentinel or terrain extracts where used;
these do not validate the synthetic population, mobility, exposure or scenario assumptions.

**Astra experiment:** Use Astra to critique visual encodings and Codex to implement a layer toggle while preserving computed access results.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## Keep physical height and thematic height separate · 35 minutes
**Learn:** combine extruded polygons, street paths, facility markers and picking in deck.gl.
The renderer consumes plain JSON generated by Python; no ipywidgets or pydeck extension is required.
Physical mode uses building heights in metres. Population mode uses **3 display metres per resident** as a
thematic chart. This is never described as actual building height. Color uses the same 0–30 minute scale in
both access scenarios, with values above 30 clipped to the top color; exact values remain in tooltips/table.
**Predict:** which change should affect color, and which should affect extrusion?


In [ ]:
from scenes import building_features, export_scene, scene_frame
c=city(); pop=allocate(c['capacity'],1200)
nodes,graph=network(blocked=[(r*11+5,r*11+6) for r in range(10)])
idx,connector=snap(c['xy'],nodes); clinic=22; added_clinic=94
baseline=(shortest(graph,[clinic])[idx]+connector)/80.
improved=(shortest(graph,[clinic,added_clinic])[idx]+connector)/80.
assert np.isfinite(baseline).all() and (improved<=baseline+1e-9).all()
geo=building_features(c,pop)
records=[]
for i,f in enumerate(geo['features']):
    records.append(dict(**f['properties'],polygon=f['geometry']['coordinates'][0],
                        baseline=float(baseline[i]),improved=float(improved[i])))
roads=[[[float(a) for a in lonlat(*nodes[u])],[float(a) for a in lonlat(*nodes[v])]] for u in graph for v,w in graph[u] if u<v]
clinics=[dict(position=[float(v) for v in lonlat(*nodes[node])],added=node==added_clinic) for node in [clinic,added_clinic]]
payload=dict(buildings=records,roads=roads,clinics=clinics,center=[float(v) for v in lonlat(1000,1000)],synthetic=True)
export_json('15_layers.json',payload)
print('Population-weighted minutes:',float(np.average(baseline,weights=pop)),'->',float(np.average(improved,weights=pop)))


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,times,label in zip(axes,[baseline,improved],['Existing clinic','Additional clinic']):
    dots=ax.scatter(*c['xy'].T,c=times,vmin=0,vmax=30,cmap='viridis_r',s=25)
    map_axes(ax,label)
fig.colorbar(dots,ax=list(axes),label='Walking minutes (same 0–30 scale)',extend='max'); plt.show()


## Use and share the interactive scene
The scene appears below in a trusted notebook output. If your notebook viewer blocks the iframe,
download the HTML from `exports/` and open it in a browser. All data are embedded, but the pinned
JavaScript/CSS renderer still downloads from a CDN; this is not fully offline. WebGL must be available.
The static plot and numerical outputs remain usable when the renderer cannot load.

Camera rotation and vertical exaggeration change the view, not the underlying data. Keep a plan view
beside a perspective view to detect occlusion. Download the notebook, scene and evidence JSON before
clearing browser storage. Do not label synthetic geometry as observed Gaborone buildings.


In [ ]:
document=export_scene('15_deckgl_scene.html','deck',payload)
display(scene_frame(document,'deck.gl population and access scenario layers'))


## Astra critique and a Codex layer task
**Astra:** provide `15_layers.json` and screenshots of both scenarios with the same camera.
Ask: “Calculate population-weighted mean travel time for each scenario. Explain whether any building's access
worsened, and identify how the population extrusion could mislead a viewer about physical height.” Check its
numbers against this notebook, not apparent column volume. Declare a 0.01-minute tolerance before the trial.

**Codex:** “Add a population-threshold filter and a served-within-15-minutes color mode in
`content/scene_templates/deck.html`. Preserve the original arrays, keep tooltips exact, and show the visible
population separately from the whole-city total. Make the legend and table reflect the active mode.”

**Acceptance:** additional-clinic travel time never exceeds baseline; population is conserved; selecting a
display metric changes no numerical data; the same color maps to the same value in both scenarios.
Try the scene with keyboard-accessible selects and compare its table with the Python export.

References: [deck.gl standalone](https://deck.gl/docs/get-started/using-standalone) and
[PolygonLayer](https://deck.gl/docs/api-reference/layers/polygon-layer).


## Extended investigation: Add timestamped street trips and explicit filtering

Construct timestamped trips along verified edges of the synthetic street graph. The scene adds a moving trip marker layer, a population filter and a served-within-15-minutes color mode. Legends and tables follow the active filter; the full-city total stays visible. The static map shows routed polylines and their origin/destination markers; the neighboring timeline shows departure and arrival times.

**Read the concept map first:** identify the input, method, evidence and limit. Predict the spatial pattern before running the comparison.


In [ ]:
from extension_tools import concept_map, export_investigation
concept_map('Add timestamped street trips and explicit filtering', ['Timed synthetic node paths', 'Move along verified edges', 'OD map + active-trip timeline', 'Straight OD lines are not routes'])
plt.show()


In [ ]:
ex_nodes,ex_graph=network(blocked=[(r*11+5,r*11+6) for r in range(10)]); ex_trips=[]
for j in range(8):
    row=j+1; path=list(range(row*11,row*11+6))+[r*11+5 for r in range(row+1,11)]+[116]+[r*11+6 for r in range(9,row-1,-1)]+list(range(row*11+7,row*11+11)); start=j*3.
    times=(start+np.arange(len(path))*200/80).tolist()
    assert np.all(np.diff(times)>0) and all(any(v==b for v,w in ex_graph[a]) for a,b in zip(path,path[1:]))
    ex_trips.append(dict(id=f'trip-{j}',node_ids=path,timestamps_min=times,path=[[float(a) for a in lonlat(*ex_nodes[node])] for node in path]))
payload['trips']=ex_trips
document=export_scene('15_deckgl_scene.html','deck',payload)
display(scene_frame(document,'deck.gl access filters and timestamped trips'))
ex_snapshot=18.; ex_active=[t for t in ex_trips if t['timestamps_min'][0]<=ex_snapshot<=t['timestamps_min'][-1]]
fig,axes=plt.subplots(1,3,figsize=(14,4))
for t in ex_trips:
    path=ex_nodes[t['node_ids']]; axes[0].plot(*path.T,alpha=.6)
    axes[0].scatter(*path[0],marker='o',s=20,c='black'); axes[0].scatter(*path[-1],marker='>',s=30,c='#c44e52')
map_axes(axes[0],'Verified street-edge trip paths')
for j,t in enumerate(ex_trips): axes[1].plot([t['timestamps_min'][0],t['timestamps_min'][-1]],[j,j],lw=5)
axes[1].axvline(ex_snapshot,c='#c44e52',ls='--'); axes[1].set(xlabel='Minutes from scenario start',ylabel='Trip ID',title='Which trips are active at 18 minutes?')
ex_cutoffs=[0,5,10,20,1000]; ex_visible=[int(sum(r['population'] for r in records if r['population']>=cutoff)) for cutoff in ex_cutoffs]
axes[2].bar([str(x) for x in ex_cutoffs],ex_visible,color='#087f8c'); axes[2].axhline(1200,c='gray',ls='--'); axes[2].set(xlabel='Minimum building population',ylabel='Visible residents',title='Filtering is not population change')
assert ex_visible[0]==1200 and ex_visible[-1]==0
fig.tight_layout(); extension_figure=fig
extension_evidence=dict(trips=ex_trips,snapshot_min=ex_snapshot,population_cutoffs=ex_cutoffs,visible_population=ex_visible,full_city_population=1200)
extension_expected=dict(active_trips=len(ex_active),full_city_population=1200)
extension_task='Count trips active at the snapshot (including endpoints) and retain the unfiltered population total. Return active_trips and full_city_population.'


In [ ]:
extension_prompt,extension_reference=export_investigation('15',extension_evidence,extension_expected,extension_task,extension_figure)
plt.show()
print('Send only 15_extension_prompt.json and optionally 15_extension_map.png to Astra. Keep the reference in this kernel.')


### Ask, check and explain the spatial result
Download `15_extension_prompt.json` and `15_extension_map.png`. Send the evidence in a fresh Astra
session with the exact task and output fields. For a separate image-only experiment, supply only the image
and task wording, record that condition, and allow clarification if the image does not contain enough information.
Never send the reference or this complete notebook for a blind trial.

Paste the unedited response below or upload a UTF-8 response file. The default is deliberately empty:
**no model trial has run**. After recording a real answer, set `extension_source='MODEL'` and complete the metadata.
The numeric tolerance is 0.01 in each requested field's declared units; labels and booleans require exact agreement.
Keep that rule fixed before examining a response. Inspect the explanation separately for unsupported claims.

**Visual reflection:** point to one map pattern, cite the numerical check that supports it, and name one thing
the visual cannot establish. Re-run one parameter change and preserve both maps. The implementation above
is available for inspection and modification; it is not a measured claim about model accuracy.


In [ ]:
from experiments import grade_answer
extension_source='NOT RUN'
extension_model=''; extension_date=''; extension_interface=''; extension_condition='text-and-map'
extension_response_file=''; extension_response_text=''
if extension_response_file:
    extension_response_text=Path(extension_response_file).read_text(encoding='utf-8')
if extension_source=='MODEL':
    assert extension_model and extension_date and extension_interface and extension_response_text, 'Record the actual model, date, interface and response'
extension_tolerances={key:.01 for key,value in extension_reference.items() if type(value) in (int,float)}
extension_grade=grade_answer(extension_response_text,extension_reference,extension_tolerances) if extension_response_text else dict(status='NOT RUN')
display(extension_grade)
export_json('15_extension_trial.json',dict(source=extension_source,model=extension_model,date=extension_date,
    interface=extension_interface,condition=extension_condition,prompt=extension_prompt,response_text=extension_response_text,
    tolerances=extension_tolerances,reference=extension_reference,grade=extension_grade))


## Save the investigation
Download the edited notebook, evidence, map and trial record from `exports/`. Browser storage does not
commit changes to GitHub. Keep data attribution and the distinction between observed data, synthetic
assumptions and actual model responses when sharing. The extended code and its checks above are part of
this notebook; use them as the starting point for your next controlled comparison.
